# Week 2: NomNaOCR pipeline-readiness audit and edit distance

Check whether inputs pair and parse correctly, why transcription characters cannot simply be zipped to detector boxes, and how edit distances measure OCR errors. Use the full read-only dataset for integrity checks and the five Week 1 triplets for visual evidence.

## 1. Clarify the problem and reproduction target

Use the papers to identify the available supervision, missing character-level artifact, and reproduction target. The supplied `detection_labels` are **reference output from a pretrained detector**: they show what localization results may look like, but are not verified ground truth and contain no character identities.

In [ ]:
import os
import unicodedata
from collections import Counter
from pathlib import Path
from statistics import median

import matplotlib.pyplot as plt
from matplotlib.patches import Patch, Polygon, Rectangle
from PIL import Image
from nom_ocr_lab import discover_page_pairs, read_detection_annotations, read_line_annotations
from nom_ocr_lab.annotations import parse_detection_annotation, parse_line_annotation

if "LAB_WORKSPACE" not in os.environ:
    raise RuntimeError("Set LAB_WORKSPACE=/data/projectgX/lab-exercise before starting Jupyter")

FULL_ROOT = Path(os.environ.get("NOMNAOCR_ROOT", "/data/shared/project2026/nomnaocr"))
SAMPLE_ROOT = Path(os.environ["LAB_WORKSPACE"]) / "datasets" / "nomnaocr-sample"

if not FULL_ROOT.is_dir():
    raise FileNotFoundError(f"Full dataset not found or unreadable: {FULL_ROOT}")
if not (SAMPLE_ROOT / "manifest.json").is_file():
    raise FileNotFoundError(
        f"Week 1 sample not found: {SAMPLE_ROOT}\n"
        "Prepare it with labs/week-01-orientation/prepare_sample.sbatch; "
        "the exact commands are in this week's README."
    )

page_pairs = discover_page_pairs(SAMPLE_ROOT)
assert len(page_pairs) == 5, f"Expected 5 sample page triplets, found {len(page_pairs)}"
print(f"Full dataset (read only): {FULL_ROOT}")
print(f"Week 1 sample: {SAMPLE_ROOT}")
print(f"Matched sample triplets: {len(page_pairs)}")

## 2. Pipeline-readiness checks

The course copy is expected to have this logical layout (the code below reports the actual tree):

```text
nomnaocr/
├── images/<document>/<page>.<image extension>
├── line_labels/<document>/<page>.txt
└── detection_labels/<document>/<page>.txt
```

`line_labels` contain quadrilateral line regions plus transcription. `detection_labels` are pretrained-detector reference output in normalized YOLO format (`class x_center y_center width height`), without character identities. Missing pairs, malformed rows, or unequal box/text counts would break later cropping and alignment.

In [ ]:
def human_bytes(number):
    value = float(number)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB"):
        if value < 1024 or unit == "TiB":
            return f"{value:.2f} {unit}"
        value /= 1024

def files_below(root):
    return sorted(path for path in root.rglob("*") if path.is_file()) if root.is_dir() else []

inventory = {}
for directory_name in ("images", "line_labels", "detection_labels"):
    paths = files_below(FULL_ROOT / directory_name)
    inventory[directory_name] = paths
    suffixes = Counter(path.suffix.lower() or "<none>" for path in paths)
    total_bytes = sum(path.stat().st_size for path in paths)
    print(f"{directory_name:>16}: {len(paths):>6} files, {human_bytes(total_bytes):>10}, formats={dict(suffixes)}")

image_keys = {path.relative_to(FULL_ROOT / 'images').with_suffix('') for path in inventory['images']}
line_keys = {path.relative_to(FULL_ROOT / 'line_labels').with_suffix('') for path in inventory['line_labels']}
detection_keys = {path.relative_to(FULL_ROOT / 'detection_labels').with_suffix('') for path in inventory['detection_labels']}
full_pairs = discover_page_pairs(FULL_ROOT)
print(f"Complete triplets: {len(full_pairs)}")
print(f"Images missing line labels: {len(image_keys - line_keys)}")
print(f"Images missing detector output: {len(image_keys - detection_keys)}")
print(f"Orphan label files: {len((line_keys | detection_keys) - image_keys)}")

### Interpret the inventory

The paper reports **2,953 pages** and **38,318 text patches**. Pages, line rows, and predicted boxes are different units.

- Are the triplets complete, and what would fail if they were not? …
- Which printed count, if any, is comparable with the paper's text-patch count? …
- What must the detection labels not be treated as? …

In [ ]:
line_files = inventory["line_labels"]
line_rows = malformed_rows = question_mark_rows = transcribed_characters = 0
malformed_examples = []
for label_path in line_files:
    for line_number, raw in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not raw.strip():
            continue
        line_rows += 1
        try:
            annotation = parse_line_annotation(raw)
        except ValueError as error:
            malformed_rows += 1
            if len(malformed_examples) < 5:
                malformed_examples.append((str(label_path), line_number, str(error)))
            continue
        question_mark_rows += "?" in annotation.text
        transcribed_characters += len(annotation.text)
print("Line labels (invalid rows corrupt geometry or target text):")
print(f"  valid={line_rows - malformed_rows}, malformed={malformed_rows}, code points={transcribed_characters}, rows with '?'={question_mark_rows}")
if malformed_examples:
    print(f"  examples={malformed_examples}")

detection_files = inventory["detection_labels"]
detection_rows = malformed_detection_rows = 0
malformed_detection_examples = []
for label_path in detection_files:
    for line_number, raw in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not raw.strip():
            continue
        detection_rows += 1
        try:
            parse_detection_annotation(raw)
        except ValueError as error:
            malformed_detection_rows += 1
            if len(malformed_detection_examples) < 5:
                malformed_detection_examples.append((str(label_path), line_number, str(error)))
print("\nPretrained-detector output (invalid boxes create bad crops):")
print(f"  valid={detection_rows - malformed_detection_rows}, malformed={malformed_detection_rows}")
if malformed_detection_examples:
    print(f"  examples={malformed_detection_examples}")

count_gaps = []
exact_count_pages = 0
for pair in full_pairs:
    character_count = sum(len(item.text) for item in read_line_annotations(pair.label_path))
    box_count = len(read_detection_annotations(pair.detection_label_path))
    count_gaps.append(box_count - character_count)
    exact_count_pages += box_count == character_count
print("\nCan transcription characters be zipped directly to detector boxes?")
print(f"  equal counts={exact_count_pages}/{len(full_pairs)} pages")
print(f"  boxes minus code points: median={median(count_gaps):.1f}, min={min(count_gaps)}, max={max(count_gaps)}")
print("  Even equal counts do not prove correct line assignment or reading order.")

## 3. Label conventions and five-page visual audit

Each line-label row is `x1,y1,...,x4,y4,text`; `?` marks missing or uncertain transcription. File order and geometry must be checked against vertical reading order.

Detection boxes are pretrained-model reference output, not verified character-level ground truth. Misses, duplicates, and ordering errors explain why direct `zip(boxes, text)` can shift every later identity and why sequence alignment is needed.

For each page below, inspect: blur/fading, ink bleed or stains, tears/cropping/skew, vertical columns versus mixed layout, marginal notes or headers, polygon tightness/overlap, reading order, `?`, punctuation, and unusual/variant characters. Do not infer quality from filename alone.

In [ ]:
sample_annotations = {}
for page_number, pair in enumerate(page_pairs, start=1):
    annotations = read_line_annotations(pair.label_path)
    detections = read_detection_annotations(pair.detection_label_path)
    sample_annotations[pair.relative_image.as_posix()] = annotations
    with Image.open(pair.image_path) as source_image:
        image = source_image.convert("RGB")

    character_count = sum(len(annotation.text) for annotation in annotations)
    uncertain_lines = sum('?' in annotation.text for annotation in annotations)
    preview = ' / '.join(annotation.text for annotation in annotations[:2])
    preview = preview if len(preview) <= 80 else preview[:77] + '...'
    print(f"\nPage {page_number}: {pair.relative_image}")
    print(f"  {image.width}x{image.height}px; lines={len(annotations)}; text={character_count}; predicted boxes={len(detections)}; uncertain lines={uncertain_lines}")
    print(f"  preview: {preview!r}")

    fig, ax = plt.subplots(figsize=(8, 11))
    ax.imshow(image)
    for line_number, annotation in enumerate(annotations, start=1):
        ax.add_patch(Polygon(annotation.points, closed=True, fill=False, edgecolor="red", linewidth=1.2))
        ax.text(annotation.points[0][0], annotation.points[0][1], str(line_number), color="yellow", fontsize=8, bbox={"facecolor": "black", "alpha": 0.6, "pad": 1})
    for detection in detections:
        left = (detection.x_center - detection.width / 2) * image.width
        top = (detection.y_center - detection.height / 2) * image.height
        ax.add_patch(Rectangle((left, top), detection.width * image.width, detection.height * image.height, fill=False, edgecolor="cyan", linewidth=0.7))
    ax.legend(handles=[Patch(facecolor="none", edgecolor="red", label="Line polygon + transcription"), Patch(facecolor="none", edgecolor="cyan", label="Pretrained-detector output (reference)")], loc="upper right")
    ax.set_title(f"Page {page_number}: {pair.relative_image} ({len(annotations)} lines, {len(detections)} predicted boxes)")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

## 4. Character-level distances and similarities

Compare an OCR hypothesis with the reference for the **same line**. This measures recognition errors and supports alignment; comparing unrelated pages does not measure OCR accuracy. Strings are sequences of Unicode code points, normalized to NFC.

**Hamming distance** (equal lengths only): $d_H(x,y)=\sum_i [x_i \ne y_i]$. It counts substitutions but cannot represent insertion or deletion.

**Levenshtein distance** uses insertion, deletion, and substitution:

$$D_{i,j}=\min\{D_{i-1,j}+1,\;D_{i,j-1}+1,\;D_{i-1,j-1}+[x_i\ne y_j]\}.$$

**Optimal String Alignment (OSA; restricted Damerau–Levenshtein)** extends Levenshtein distance by allowing the transposition of two adjacent symbols at a cost of 1. Let \(D_{i,j}\) denote the minimum edit cost between the first \(i\) and \(j\) symbols of strings \(s\) and \(t\), respectively. When the last two symbols are swapped, i.e.,

$$
s_i=t_{j-1},\qquad s_{i-1}=t_j,
$$

an additional transition is considered:

$$
D_{i,j}=\min\left(D_{i,j},\;D_{i-2,j-2}+1\right).
$$

For example, transforming `AB` into `BA` requires only one transposition, giving \(D_{2,2}=1\). Unlike unrestricted Damerau–Levenshtein distance, OSA restricts overlapping edits, so the same substring cannot be edited more than once in a transformation.

**LCS distance** derives from the longest common subsequence: $d_{LCS}(x,y)=|x|+|y|-2|LCS(x,y)|$. It models insertions and deletions, not substitution as a single edit.

For Levenshtein/OSA we use $s=1-d/\max(|x|,|y|)$. For LCS we use $s=2|LCS|/(|x|+|y|)$. Character error rate is $CER=d_{Lev}(reference,hypothesis)/|reference|$.

In [ ]:
def hamming_distance(left, right):
    if len(left) != len(right):
        raise ValueError("Hamming distance requires equal-length sequences")
    return sum(a != b for a, b in zip(left, right))

def levenshtein_distance(left, right):
    # Two rows are enough when only the final distance is required.
    previous = list(range(len(right) + 1))
    for i, left_item in enumerate(left, start=1):
        current = [i]
        for j, right_item in enumerate(right, start=1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (left_item != right_item)))
        previous = current
    return previous[-1]

def osa_damerau_levenshtein_distance(left, right):
    table = [[0] * (len(right) + 1) for _ in range(len(left) + 1)]
    for i in range(len(left) + 1):
        table[i][0] = i
    for j in range(len(right) + 1):
        table[0][j] = j
    for i in range(1, len(left) + 1):
        for j in range(1, len(right) + 1):
            table[i][j] = min(table[i - 1][j] + 1, table[i][j - 1] + 1, table[i - 1][j - 1] + (left[i - 1] != right[j - 1]))
            if i > 1 and j > 1 and left[i - 1] == right[j - 2] and left[i - 2] == right[j - 1]:
                table[i][j] = min(table[i][j], table[i - 2][j - 2] + 1)
    return table[-1][-1]

def lcs_length(left, right):
    previous = [0] * (len(right) + 1)
    for left_item in left:
        current = [0]
        for j, right_item in enumerate(right, start=1):
            current.append(previous[j - 1] + 1 if left_item == right_item else max(previous[j], current[j - 1]))
        previous = current
    return previous[-1]

def lcs_distance(left, right):
    return len(left) + len(right) - 2 * lcs_length(left, right)

def normalized_edit_similarity(left, right, distance=levenshtein_distance):
    denominator = max(len(left), len(right))
    return 1.0 if denominator == 0 else 1.0 - distance(left, right) / denominator

def lcs_similarity(left, right):
    denominator = len(left) + len(right)
    return 1.0 if denominator == 0 else 2.0 * lcs_length(left, right) / denominator

assert hamming_distance("Nôm", "Nơm") == 1
assert levenshtein_distance("kitten", "sitting") == 3
assert osa_damerau_levenshtein_distance("ab", "ba") == 1
assert lcs_length("ABCBDAB", "BDCABA") == 4
assert normalized_edit_similarity("", "") == 1.0
print("All distance sanity checks passed.")

In [ ]:
# Derive controlled OCR-like errors from a real line transcription.
reference = max(
    (item.text for annotations in sample_annotations.values() for item in annotations),
    key=len,
)
reference = unicodedata.normalize("NFC", reference)
i = len(reference) // 2
replacement = "□" if reference[i] != "□" else "?"
hypotheses = {
    "exact copy": reference,
    "substitution (recognizer confusion)": reference[:i] + replacement + reference[i + 1:],
    "deletion (missed character/box)": reference[:i] + reference[i + 1:],
    "insertion (duplicate/extra character)": reference[:i] + reference[i] + reference[i:],
    "transposition (ordering error)": reference[:i] + reference[i + 1] + reference[i] + reference[i + 2:],
}

print(f"Reference ({len(reference)} code points): {reference}")
print("\nOCR-style transcription comparisons")
print(f"{'scenario':40} {'Hamming':>8} {'Lev':>5} {'CER':>7} {'Normalized Levenshtein':>23} {'OSA':>5} {'LCS':>7}")
for scenario, hypothesis in hypotheses.items():
    hamming = str(hamming_distance(reference, hypothesis)) if len(reference) == len(hypothesis) else "n/a"
    lev = levenshtein_distance(reference, hypothesis)
    print(f"{scenario:40} {hamming:>8} {lev:5d} {lev / len(reference):7.3f} {normalized_edit_similarity(reference, hypothesis):23.3f} {osa_damerau_levenshtein_distance(reference, hypothesis):5d} {lcs_similarity(reference, hypothesis):7.3f}")


## 5. Interpret the metrics

- Match each simulated error to a detector, recognizer, or reading-order failure. …
- Why does Hamming fail for misses or duplicates, and why does OSA distinguish the swap? …
- Why is Levenshtein-based CER useful for OCR? What does LCS add? …
- How should `?`, punctuation, normalization, and historical variants be handled before scoring? …

Return to the synthesis statement in Section 1. Revise it if the dataset evidence changed your understanding of the available or missing supervision.